# Ensembl REST

![Ensembl REST](https://proto-bio.github.io/proto-assets/images/tool/ensembl/hero.png)

Five tools share the `ensembl` toolkit, one per endpoint family:

- **`ensembl-lookup`** — gene lookup by Ensembl gene ID **or** gene symbol.
- **`ensembl-sequence`** — DNA / cDNA / CDS / protein sequence retrieval by Ensembl ID.
- **`ensembl-overlap`** — features overlapping a region (genes, exons, regulatory elements, motifs, variants).
- **`ensembl-xrefs`** — cross-references from an Ensembl ID to external databases.
- **`ensembl-vep`** — variant-effect prediction from HGVS notation.

All five call `rest.ensembl.org` (GRCh38) by default; `assembly="GRCh37"` routes to `grch37.rest.ensembl.org`.

In [ ]:
from proto_tools.tools.database_retrieval import (
    EnsemblLookupConfig,
    EnsemblLookupInput,
    EnsemblOverlapConfig,
    EnsemblOverlapInput,
    EnsemblSequenceConfig,
    EnsemblSequenceInput,
    EnsemblVEPInput,
    EnsemblXrefsInput,
    run_ensembl_lookup,
    run_ensembl_overlap,
    run_ensembl_sequence,
    run_ensembl_vep,
    run_ensembl_xrefs,
)

In [ ]:
from proto_tools.utils.notebook_docs import display_api_reference

## ensembl-lookup — gene lookup with transcript expansion

In [ ]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_lookup")

In [ ]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_lookup")

In [ ]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_lookup")

In [ ]:
out = run_ensembl_lookup(EnsemblLookupInput(symbol="BRCA1"), EnsemblLookupConfig(expand=True))
assert out.success
gene = out.result
print(f"{gene.display_name} ({gene.id}): {len(gene.Transcript)} transcripts; canonical={gene.canonical_transcript}")

## ensembl-sequence — protein sequence of the canonical transcript

In [ ]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_sequence")

In [ ]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_sequence")

In [ ]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_sequence")

In [ ]:
canonical_id = gene.canonical_transcript.split(".")[0]
seq_out = run_ensembl_sequence(
    EnsemblSequenceInput(ensembl_id=canonical_id),
    EnsemblSequenceConfig(sequence_type="protein"),
)
print(f"{seq_out.results[0].id}: {len(seq_out.results[0].seq)} aa")

## ensembl-overlap — regulatory features at the BRCA1 locus

In [ ]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_overlap")

In [ ]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_overlap")

In [ ]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_overlap")

In [ ]:
overlap_out = run_ensembl_overlap(
    EnsemblOverlapInput(ensembl_id=gene.id),
    EnsemblOverlapConfig(overlap_feature="regulatory"),
)
print(f"{len(overlap_out.result)} regulatory features overlapping {gene.display_name}:")
for record in overlap_out.result[:5]:
    print(f"  {record.feature_type} {record.id} ({record.seq_region_name}:{record.start}-{record.end})")

## ensembl-xrefs — cross-reference Ensembl → UniProt

In [ ]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_xrefs")

In [ ]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_xrefs")

In [ ]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_xrefs")

In [ ]:
# Cross-reference the canonical protein to its reviewed Swiss-Prot accession (BRCA1 -> P38398).
protein_id = seq_out.results[0].id
xrefs_out = run_ensembl_xrefs(EnsemblXrefsInput(ensembl_id=protein_id))
swissprot = next(x for x in xrefs_out.result if x.dbname == "Uniprot/SWISSPROT")
print(f"Swiss-Prot accession for {gene.display_name}: {swissprot.primary_id} (via {protein_id})")

## ensembl-vep — Variant Effect Prediction

HGVS forms supported: genomic (`9:g.22125504G>C`), coding (`ENST00000357654:c.5074G>A`), or protein (`ENSP00000418960:p.Tyr124Cys`).

In [ ]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_vep")

In [ ]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_vep")

In [ ]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_vep")

In [ ]:
vep = run_ensembl_vep(EnsemblVEPInput(hgvs="9:g.22125504G>C"))
top = vep.consequences[0]
print(f"{top.allele_string}: {top.most_severe_consequence}, {len(top.transcript_consequences)} transcripts affected")

## GRCh37 — same gene, different coordinates

Set `assembly="GRCh37"` for legacy/clinical datasets pinned to the older assembly.

In [ ]:
grch37 = run_ensembl_lookup(
    EnsemblLookupInput(ensembl_id=gene.id),
    EnsemblLookupConfig(assembly="GRCh37"),
)
print(f"GRCh38 start: {gene.start} -> GRCh37 start: {grch37.result.start}")